| # | Layout | Files | Shape |
|---|---|---|---|
| **A** | **Case Record Form** (the official CRF) | พิมาย, ท่าวุ้ง, กงหรา, หูแร่ ×3, อุ้มผาง | one row per patient, 22–24 fixed columns, medication as free text |
| **A′** | **CRF, site variants** | บางระกำ, PCC คลองศาลา | same concept, grouped two-row headers; PCC has one column per drug per visit and monthly anthropometry |
| **B** | **รพ.สต. summary workbook** | 13 sub-district health-promoting hospitals | sheets `เบาหวาน` / `ความดัน` / `กลุ่มเสี่ยง`; two-row header with a `ก่อน` block and a `หลัง` block; one column per drug holding a dose string |
| **C** | **Relational HIS export** | รพ.บ้านตาขุน | **four sheets** keyed on `รหัสประจำตัวผู้เข้าร่วม`; follow-up values, and per-visit costs, packed into single cells |
| **D** | **Google-Forms export** | "for edit ข้อมูลคนไข้โครงการ 2" | baseline + 3-month HbA1c, tablet counts, no follow-up medication column |

Only the DM sheets are read. The `ความดัน` (hypertension) and `กลุ่มเสี่ยง`
(pre-diabetes) sheets in layout B are a different population and a different
programme arm, so they are left out of this dataset.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from src import dmr_etl, dmr_parse

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RAW = Path.cwd().parent / "data" / "case_report"
PROC = Path.cwd().parent / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

files = sorted(RAW.glob("*.xls*"))


def find_file(*keywords):
    """Locate a source workbook by keyword, so renames do not break the notebook."""
    for f in files:
        if all(k in f.name for k in keywords):
            return f
    raise FileNotFoundError(keywords)


print(f"{len(files)} workbooks found\n")
for f in files:
    print(f"  {dmr_etl.classify_file(str(f)):11s}  {f.name}")

print("\nSource files were renamed to their facility name; the originals are "
      "recorded in\n  data/case_report/_filename_map.csv")

---
## 1 · The three parsing problems

Before any file can be read, three kinds of messiness have to be handled. These
are the parsers in `src/dmr_parse.py`; the cells below show them working so the
rules can be checked against what is actually in the cells.

### 1.1 Dates — Buddhist Era, Gregorian, and a two-digit-year trap

Sites mixed BE and CE freely. There is also a systematic Excel artefact: a cell
typed as `9/10/68` (meaning 9 Oct **2568** BE) is stored by Excel as **1968**.
The `[DM remission]_ข้อมูลผู้ป่วย` workbook has its whole enrolment column like
this. The rule applied is:

* year ≥ 2400 → Buddhist Era, subtract 543 → `2568 → 2025`
* 1940 ≤ year ≤ 1999 → truncated two-digit BE year, add 57 → `1968 → 2025`
* otherwise → already Gregorian

In [ ]:
examples = ["2568-01-15", "03/02/2558", "12-05-2566", "2567",
            "Dx.HT 20/03/2564", "1 ส.ค. 67", "2016-01-20",
            pd.Timestamp("1968-01-01"), "รับยาที่อื่น พ.ศ. 2560"]
pd.DataFrame({"raw": [str(e) for e in examples],
              "parsed": [dmr_parse.parse_date(e) for e in examples]})

### 1.2 Numbers — values with comments attached

Lab cells often carry the sample date or a note. Each measurement is also held
to a physiological window (`dmr_parse.RANGES`), so a typo such as an FBS of
`105115` becomes missing instead of poisoning a mean.

In [ ]:
examples = ["118  (2/8/67)", "6.5 %", ">200", "ไม่ได้มาเจาะ", "105115",
            "7,2", "650", "-"]
pd.DataFrame({
    "raw": examples,
    "as FBS (30–600)":   [dmr_parse.to_num_r(e, "fbs") for e in examples],
    "as HbA1c (3.5–20)": [dmr_parse.to_num_r(e, "hba1c") for e in examples],
})

### 1.3 Medication — two completely different encodings

**(a) One column per drug, holding a dose string** (layout B, PCC).
`parse_dose` converts it to *tablets per day*, so `1x2` = 1 tablet twice daily
= 2, and `2-0-1` = a morning–noon–evening schedule = 3.

The critical distinction is between **0** and **missing**: `off`, `0`, `-`,
`ไม่มี` mean *not on the drug*; a blank cell or `ไม่ทราบ` means *not recorded*.
Collapsing those two would silently invent drug cessations.

In [ ]:
examples = ["1x1", "1*2", "2x2", "2-0-1", "1/2x1", "1 1/2x1", "0.5x1 hs",
            "18-0-12", "off", "0", "-", "ไม่มี", "Diet", None, "ไม่ทราบ",
            "sim1xhs", "201"]
pd.DataFrame({"raw": [str(e) for e in examples],
              "tabs/day": [dmr_parse.parse_dose(e) for e in examples]})

**(b) One free-text column holding the whole regimen** (layout A, C).
`parse_regimen` matches drug names against a class dictionary and returns the
set of classes the patient is **on**.

Two behaviours matter:

* **Segment-level `OFF`.** `MFM(500)2x2 / OFF Glipizide(5)2X2` means metformin
  continues and glipizide stopped — not that everything stopped. The string is
  split on `,` `/` `;` `+` and an `OFF` inside a segment kills only that drug.
* **`ยังไม่ปรับยา` ("not adjusted yet")** is returned as `no_change`, and the
  loader carries the previous timepoint's regimen forward rather than reading
  it as missing.

In [ ]:
examples = [
    "Metformin (500) 2*2 po pc, Glipizide (5) 1x1",
    "MFM(500)2x2/Glipizide(5)2X2",
    "OFF Glipizide(5)2X2",
    "MFM(500)1x2/OFF Glipizide(5)1X2",
    "NPH INSULIN GLI-PIZIDE TAB 5 mg. METFORMIN 850 mg.",
    "Empagliflozin(Jardiance) | METFORMIN(Glucophage)",
    "ไม่มียาDM case diet conttrol",
    "OFF ยา",
    "ยังไม่ปรับยา",
    "ไม่มี",
    None,
]
rows = []
for e in examples:
    r = dmr_parse.parse_regimen(e)
    rows.append({"raw": (str(e)[:46] if e else "(blank)"),
                 "status": r["status"], "on_any": r["on_any"],
                 "n_classes": r["n_classes"],
                 "classes": ", ".join(sorted(r["classes"])) or "—"})
pd.DataFrame(rows)

### 1.4 The บ้านตาขุน export — four sheets that have to be read together

This workbook is relational: four sheets joined on `รหัสประจำตัวผู้เข้าร่วม`.
Two of them pack a whole time series into a single cell, and the sheets only
make sense when read against each other.

| Sheet | Content | Packing |
|---|---|---|
| `ข้อมูลพื้นฐาน` | demographics, diagnosis date, baseline values, clinic membership status | one row per patient |
| `ข้อมูล Lab หลังเข้าร่วม` | follow-up weight / waist / BMI / HbA1c / FBS | `date \| value \| date \| value …` |
| `ข้อมูลยาลดน้ำตาล` | glucose-lowering drugs dispensed before / after the index date | `drug \| drug \| drug` (a **list**, no dates) |
| `ข้อมูลค่าใช้จ่าย` | per-visit costs | `date \| lab cost \| drug cost \| …` — **triples**, not pairs |

`_unpack_series` takes a `stride` argument for exactly this reason: 2 for the
lab sheet, 3 for the cost sheet. Series are sorted by date, so `[-1]` is the
most recent measurement and `[0]` the earliest.

In [ ]:
f = find_file("บ้านตาขุน")
xl = dmr_etl.open_workbook(f)
lab = xl.parse("ข้อมูล Lab หลังเข้าร่วม")
cost = xl.parse("ข้อมูลค่าใช้จ่าย")

a1c_col = [c for c in lab.columns if "hba1c" in str(c).lower()][0]
raw_cell = lab[a1c_col].dropna().iloc[2]
print("RAW HbA1c cell:\n ", raw_cell, "\n")

series = dmr_etl._unpack_series(raw_cell, stride=2)
print("unpacked (stride=2):")
for dt, v in series:
    print(f"   {str(dt.date()) if pd.notna(dt) else '?':>12}   {v}")
print(f"\n  -> hba1c_post takes the LAST value: {series[-1][1]}")

print("\nRAW cost cell:\n ", str(cost.iloc[0, 2])[:120], "...")
ctrip = dmr_etl._unpack_series(cost.iloc[0, 2], stride=3)
print("\nunpacked (stride=3)  date | lab cost | drug cost:")
for dt, lc, dc in ctrip[:4]:
    print(f"   {dt.date()}   lab {lc:>8.1f}   drug {dc:>8.1f}")

#### The medication sheet is a dispensing *list*, and a blank cell is ambiguous

`ยาลดน้ำตาลหลังเข้าร่วมโครงการ` lists every glucose-lowering drug dispensed
after the index date. It has no dates, so on its own a **blank cell means
either "no drug dispensed" (a genuine stop) or "this patient was never
followed up"** — and those are opposite conclusions.

The other three sheets resolve it. A patient with post-index laboratory
results or post-index visits demonstrably kept attending the clinic, so a
blank drug list for *them* means no glucose-lowering drug was dispensed. With
no post-index contact at all, the cell stays missing.

**This inference is checkable against pharmacy cost, which is an independent
sheet.** If the rule is right, the "blank but still attending" group should
have near-zero drug spending.

In [ ]:
# load this one workbook on its own so the check does not depend on the full
# pipeline having run yet
bt = dmr_etl.add_derived(dmr_etl.load_bantakhun(str(f)))

bt["group"] = np.select(
    [bt["med_status_pre"].eq("on_drug") & bt["med_status_post"].eq("on_drug"),
     bt["med_status_pre"].eq("on_drug") & bt["med_status_post"].eq("no_drug_inferred"),
     bt["med_status_pre"].eq("on_drug") & bt["med_status_post"].eq("unknown")],
    ["on drugs before and after",
     "pre-drug, post BLANK + still attending  -> inferred STOPPED",
     "pre-drug, post BLANK + no contact       -> left missing"],
    default="no drug recorded before")

chk = bt.groupby("group").agg(
    n=("group", "size"),
    post_visits=("n_post_visits", "mean"),
    drugcost_per_visit=("drugcost_post", "mean"),
    drugcost_last_visit=("drugcost_post_last", "mean"),
    pct_zero_drugcost_at_last_visit=("zero_drugcost_last", lambda s: 100 * s.mean()),
).round(1)
chk

The check holds. Patients whose drug list is blank **but who kept attending**
average roughly a tenth of the drug spending of those still on treatment, and
most have **zero** glucose-lowering spend at their last visit. Patients with no
post-index contact have no cost data at all, which is why they are left
missing rather than counted as stops.

One caveat carried forward: `ค่ายา` is *total* drug cost, so a patient who
stopped their diabetes drugs but stayed on antihypertensives still shows a
non-zero figure. The agreement is therefore strong but not perfect, and §1.5
quantifies it.

### 1.5 What this export supports that the others do not

Reading all four sheets yields three things no other workbook has:

* **Real follow-up duration** — the lab series carries dates, so the interval
  between the index date and the last measurement is known per patient
  (median ≈ 28 months) instead of being an unrecorded "post".
* **Cost** — lab and drug spending per visit, before and after, which makes a
  cost outcome possible.
* **Clinic membership status** — `clinic_member_status_name` records death,
  loss to follow-up and transfer out explicitly, so those patients can be
  excluded from a follow-up outcome instead of silently diluting it.

In [ ]:
print("follow-up months (from the dated lab series):")
print(bt["followup_months"].describe().round(1).to_string())

print("\nclinic membership status:")
print(bt["member_status"].value_counts(dropna=False).to_string())
print("\n-> `active_in_care` keeps only ยังรักษาอยู่ "
      f"({int(bt['active_in_care'].sum())} of {len(bt)});"
      " the rest are dead, disengaged or transferred out.")

print("\ncare setting (note field):")
print(bt["care_note"].value_counts(dropna=False).head(5).to_string())

---
## 2 · One damaged workbook

`อุ้มผาง ... ฉบับเเก้ไข้.xlsx` will not open: the first 31 bytes of the zip are
corrupted, so the `[Content_Types].xml` member cannot be inflated. Every other
member — including the worksheet and the shared-string table — is intact.

That member is pure boilerplate, so `dmr_etl.open_workbook` regenerates it in
memory and rebuilds the archive. **No cell data is altered**; only the manifest
is replaced. Without this, 40 patients would be lost.

In [ ]:
import zipfile
bad = find_file("อุ้มผาง")

try:
    pd.ExcelFile(bad)
except Exception as exc:
    print("pandas alone :", type(exc).__name__, "—", exc)

print("corrupt member:", zipfile.ZipFile(bad).testzip())
xl = dmr_etl.open_workbook(bad)
print("after repair  :", xl.sheet_names, "→", xl.parse(xl.sheet_names[0]).shape)

---
## 3 · Harmonised schema

Every loader emits the same columns, so combining is a concatenation. Column
names are English and lower-case; the original Thai text is kept in the
`*_raw` fields so any derived value can be traced back to the cell it came
from.

In [ ]:
groups = {
    "provenance":   dmr_etl.SCHEMA[0:5],
    "demographics": dmr_etl.SCHEMA[5:9],
    "history":      dmr_etl.SCHEMA[9:20],
    "anthropometry":dmr_etl.SCHEMA[20:27],
    "glycaemia":    dmr_etl.SCHEMA[27:35],
    "other labs":   dmr_etl.SCHEMA[35:43],
    "medication":   dmr_etl.SCHEMA[43:],
}
for g, cols in groups.items():
    print(f"{g:14s} ({len(cols):2d})  " + ", ".join(cols))

### Fields that had to be reconstructed rather than read

| Field | Where it comes from |
|---|---|
| `sex` for layout B | The รพ.สต. template has **no sex column**. Sex is taken from the Thai courtesy title on the name (`นาย` → male, `นาง`/`น.ส.` → female). The female check runs first because `นางสาว` also starts with `นาง`. |
| `bmi_pre` / `bmi_post` for layouts B & D | Not recorded; computed from height and weight, then range-checked. |
| `*_post` for layout C | Unpacked from the `date \| value \| date \| value` strings, sorted by date, last value taken. |
| `facility` for layout B | Parsed out of the A1 title. Some sites pasted the risk-group title by mistake, so any `ข้อมูล…เบาหวาน` preamble is stripped. อุ้มผาง wrote a facility *code* (`11243`); numeric values fall back to the site in the filename. |
| `dm_duration_y` | `enroll_date − dx_date`, kept only when 0–50 years. **Unavailable for layout B and D — those templates carry no dates at all.** |

---
## 4 · Run the pipeline

In [ ]:
master, audit, dropped = dmr_etl.build_master(RAW)
print(f"\ncombined: {master.shape[0]} patients × {master.shape[1]} columns")

### 4.1 Per-file audit

The point of this table is to catch a loader that silently read nothing. Every
file should contribute roughly the number of rows its sheet has, and the
`n_*` columns show how much of each key field actually survived parsing.

In [ ]:
audit.style.background_gradient(cmap="Blues", subset=["n_rows"]).format(precision=0)

### 4.2 De-duplication

Site หูแร่ submitted **three overlapping workbooks** — an original plus two
`เพิ่มเติม` ("additional") versions that re-send the original rows. Left alone
those patients would be counted up to three times.

Patients are matched on `facility + sex + age + baseline BMI + waist + FBS +
HbA1c`. Names are not present in every layout and are not needed: within one
facility, the chance that two different patients share sex, age and four
baseline measurements to the recorded precision is negligible. The copy kept is
the one from the file contributing the most rows — i.e. the most complete
resubmission.

Rows with fewer than two informative baseline values are never treated as
duplicates, so sparse records are not lost to a false match.

In [ ]:
print(f"{len(dropped)} duplicate rows removed\n")
print(dropped["source_file"].value_counts().to_string())

hurae = master[master["facility"].astype(str).str.contains("หูแร่", na=False)]
print(f"\nหูแร่ after de-duplication: {len(hurae)} patients "
      f"(submitted across {dropped['source_file'].str.contains('หูแร่').sum() + len(hurae)} rows)")

### `cohort = "survey_2023"` — 214 patients
The Google-Forms export, filed as "โครงการ 2" and collected 22–24 Aug 2023
across the same Kranuan network as the 13 รพ.สต. workbooks.

**It is a separate round, not a duplicate.** Matching on HN and on name both
give about **8 %** overlap with those workbooks (17 and 18 patients
respectively, agreeing closely — so this is a real difference, not a matching
failure). HN links cleanly once normalised: the รพ.สต. files store `00N0NNN`,
the survey `N0NNN.0`.

Its medication is a **count** of tablets and injections at baseline and at
3 months, not a named regimen, and `med_outcome_basis` records that. It is
analysed at its own 3-month anchor in notebook `02` §5.3.

In [ ]:
# how much do the survey and the รพ.สต. workbooks actually overlap?
import re

def hn_norm(x):
    s = dmr_parse.clean_text(x)
    if s is None:
        return None
    s = re.sub(r"\.0+$", "", s.strip())    # N0NNN.0 -> N0NNN
    s = re.sub(r"\D", "", s).lstrip("0")   # 00N0NNN -> N0NNN
    return s or None

srv = master[master["cohort"] == "survey_2023"]
rph = master[master["source_family"] == "rphc"]
a = {h for h in srv["local_id"].map(hn_norm) if h}
b = {h for h in rph["local_id"].map(hn_norm) if h}

print(f"survey HNs      : {len(a)}")
print(f"รพ.สต. HNs       : {len(b)}")
print(f"shared          : {len(a & b)}  ({100*len(a & b)/max(len(a), 1):.0f}% of the survey)")
print("\n-> same 13 facilities, largely different patients: a second round, "
      "not a re-submission.")

In [ ]:
print(master.groupby("cohort").agg(
    patients=("patient_uid", "size"),
    sites=("facility", "nunique"),
    basis=("med_outcome_basis", "first"),
    med_outcome_valid=("med_outcome_valid", "first"),
).to_string())

print("\nDrug cessation, at whichever anchor each cohort supports:")
print("  6 months:")
print(master.groupby("cohort")["drug_stopped_m6"]
      .agg(eligible="count", stopped="sum").to_string())
print("  3 months:")
print(master.groupby("cohort")["drug_stopped_m3"]
      .agg(eligible="count", stopped="sum").to_string())

print("\nRegistry medication status after the index date:")
print(master[master["cohort"] == "registry"]["med_status_post"]
      .value_counts(dropna=False).to_string())
print("  no_drug_inferred = blank drug list + evidence of continued attendance")

### 5.1 Last observed regimen

Many CRF sites left `ยาลดระดับน้ำตาลหลังจบจากโครงการ` blank because the
programme had not finished, but did fill in the during-programme column.
Restricting to the end-of-programme column alone would discard those patients.

So the primary outcome uses the **last observed regimen** — the
end-of-programme column where present, otherwise the during-programme column —
and `med_last_timepoint` records which was used. A stricter
`drug_stopped_strict`, using the end-of-programme column only, is carried as a
sensitivity analysis in notebook `02`.

### 5.2 Time-bound outcomes — the 6-month anchor

Outcomes are defined at **6 months after enrolment** rather than at each
patient's last recorded value. "Last observed" conflates a patient measured at
3 months with one measured at 12, and made the programme and registry cohorts
incomparable because their follow-up lengths differ by design.

The workbooks support that anchor very unevenly, so every 6-month value carries
a **tier** recording how its timing was established (`M6_SOURCE` in
`dmr_etl.py`). The assumption is a column in the data, not a footnote.

| Tier | Meaning | Source |
|---|---|---|
| `measured` | a real measurement date, accepted within **±3 months** of the target | รพ.บ้านตาขุน — the only workbook with dates |
| `labelled` | a column the template explicitly labels *N months since enrolment* | CRF `FBS ระหว่างเข้าร่วมโครงการ 6 เดือน`, บางระกำ `FBS 6 เดือน`, PCC `6 mo.` |
| `assumed` | a single unlabelled follow-up column, taken to be ≈ 6 months | รพ.สต. `หลัง` block; CRF mid-programme columns |

**Two traps the tiering exists to catch.**

1. The CRF's `หลังจบโครงการ 6 เดือน` is anchored to the programme's **end**,
   not to enrolment — a different clock. It is *not* used; the mid-programme
   column is used instead, at the `assumed` tier.
2. รพ.พิมาย's 22-column CRF has only one *unlabelled* mid-programme FBS column
   where the 24-column files have an explicit `6 เดือน` one. Both land in the
   same field, so without the tier its 100 patients would silently inherit a
   timing quality they do not have.

In [ ]:
M6 = {"hba1c": "hba1c_m6", "fbs": "fbs_m6", "bmi": "bmi_m6",
      "waist": "waist_m6", "medication": "on_any_m6"}

cov = []
for lab, col in M6.items():
    for coh in ["program", "registry"]:
        sub = master[master["cohort"] == coh]
        for tier in ["measured", "labelled", "assumed"]:
            n = int((sub[f"{col}_tier"] == tier).sum())
            if n:
                cov.append({"measure": lab, "cohort": coh, "tier": tier, "n": n})
cover = (pd.DataFrame(cov).pivot_table(index=["measure", "cohort"], columns="tier",
                                       values="n", fill_value=0)
         .reindex(columns=["measured", "labelled", "assumed"], fill_value=0))
cover["total"] = cover.sum(axis=1)
cover

In [ ]:
# the fix from trap 2, visible: the two CRF versions land in different tiers
crf = master[master["source_family"] == "crf"]
print(crf.groupby([crf["source_file"].str.slice(0, 40), "fbs_m6_tier"])["fbs_m6"]
      .count().to_string())

print("\nregistry: how close the 'measured' values actually fall to 6 months")
print(master.loc[master["cohort"] == "registry", "m6_offset_months"]
      .describe().round(2).to_string())

**Medication has no `measured` tier anywhere.** The registry's drug sheet is an
undated dispensing list, so there is no way to say what a patient was taking at
month 6 specifically — **drug cessation at 6 months does not exist for the
registry**. It is available only in the programme cohort, and mostly at the
`assumed` tier. Notebook `02` keeps the last-observed cessation figure for the
registry and never compares the two as if they were the same quantity.

Both definitions are kept in the master table: `*_m6` for the time-bound
outcomes and `*_post` / `*_last` for the previous ones, so the effect of the
definition can be seen rather than assumed.

In [ ]:
prog = master[master["cohort"] == "program"]
print(prog["med_last_timepoint"].value_counts(dropna=False).to_string())
print("\ndrug-cessation analysis set:")
print(f"  on a glucose-lowering drug at baseline, follow-up regimen recorded: "
      f"{int(prog['drug_stop_eligible'].sum())}")
print(f"  of whom stopped all drugs: {int(prog['drug_stopped'].sum())}")

---
## 6 · Completeness of the combined table

Missingness here is overwhelmingly **structural** — a template either has a
field or it does not — rather than patient-level non-response. That matters for
the modelling: it means the missingness is a property of the site's paperwork,
not of the patient, and it is why the regression in notebook `02` is built on
the fields every layout carries.

In [ ]:
key = ["sex", "age", "scheme", "education", "dx_date", "enroll_date",
       "dm_duration_y", "bmi_pre", "bmi_post", "waist_pre", "waist_post",
       "fbs_pre", "fbs_last", "hba1c_pre", "hba1c_last",
       "on_any_pre", "on_any_last", "tabs_pre", "tabs_post"]

comp = (master.groupby("source_family")[key]
        .apply(lambda x: (x.notna().mean() * 100).round(0)).T)
comp["ALL"] = (master[key].notna().mean() * 100).round(0)
comp.style.background_gradient(cmap="Blues", vmin=0, vmax=100).format("{:.0f}%")

### Range checks

A final look at whether the harmonised values are physiologically sensible. If
a parser had mis-read a column — a dose read as a lab value, a BE year left
unconverted — it would show up here as an impossible minimum or maximum.

In [ ]:
num = ["age", "dm_duration_y", "height_cm", "weight_pre", "bmi_pre", "bmi_post",
       "waist_pre", "waist_post", "fbs_pre", "fbs_last", "hba1c_pre",
       "hba1c_last", "tabs_pre", "tabs_post", "n_class_pre", "n_class_last"]
master[num].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
# enrolment dates should sit inside the programme window
ed = pd.to_datetime(master["enroll_date"], errors="coerce")
print("enrolment date range:", ed.min().date(), "→", ed.max().date())
print("\nby cohort:")
print(master.assign(yr=ed.dt.year).groupby("cohort")["yr"]
      .value_counts().unstack(fill_value=0).to_string())

---
## 7 · Write the outputs

In [ ]:
master.to_csv(PROC / "dmr_master.csv", index=False)
audit.to_csv(PROC / "dmr_source_audit.csv", index=False)
dropped.to_csv(PROC / "dmr_dropped_duplicates.csv", index=False)

print("written:")
for f in sorted(PROC.glob("*.csv")):
    print(f"  {f.name:34s} {f.stat().st_size/1024:8.0f} KB")
print(f"\nmaster: {master.shape[0]} patients × {master.shape[1]} columns")
print("→ continue in 02_analysis_effectiveness.ipynb")

---
## Known limitations of the combined dataset

Carried forward into notebook `02` and into anything written from it.

1. **No patient identifiers across files.** De-duplication is within-facility
   and measurement-based. A patient who moved between a รพ.สต. and its parent
   hospital would appear twice, and could not be detected.
2. **`dm_duration_y`, benefit scheme and comorbidity are missing for the entire
   รพ.สต. layout** — those templates have no date or diagnosis fields. Any model
   using them is restricted to the hospital-based sites, which is a different
   case mix, not a random subsample.
3. **Follow-up length is not recorded** in most templates. "Post" means
   "the last value the site wrote down", which is 3–12 months depending on the
   site and sometimes on the patient. Changes are therefore not anchored to a
   fixed interval.
4. **No control group.** Every design here is before-and-after within the same
   patients. Regression to the mean, secular trends in diabetes care, and
   selection of motivated patients into the programme are all uncontrolled, and
   none of the effect estimates can be read as causal.
5. **Remission cannot be verified against the consensus definition.** ADA/EASD
   2021 requires HbA1c < 6.5 % measured **at least 3 months after** drugs stop.
   The date drugs stopped is not recorded, so the figure reported is an
   **upper bound**.
6. **Medication detail is lossy in the CRF layout.** Free text gives drug
   *classes* reliably but daily dose only sometimes, so `tabs_pre`/`tabs_post`
   exist only for the รพ.สต. and PCC layouts.